# AGC ttbar — coffea-casa

Same workflow, same analysis code - the facility changes to `CoffeaCasaFactory`, which connects to coffea-casa's pre-configured Dask cluster (`tls://localhost:8786`) and ships the analysis code and required packages to the workers. Run this notebook **inside a coffea-casa JupyterLab session**.

> Coffea-casa note: if you install custom packages on workers, use a **fixed** number of workers (min == max). With adaptive scaling, a freshly added worker lacks the packages and can stall the run.

This notebook also demonstrates **chunk-level fault tolerance**: `with_failure=True` corrupts one file URL on purpose.

In [1]:
import sys

from coffea_workflow import Step, Workflow, Fileset, Analysis, Plotting, RunConfig, ExecutorConfig, run
from coffea_workflow import facilities
from ttbar_analysis import get_fileset, run_analysis, plotting_1

In [2]:
step_fileset = Step(
    name="Fileset_ttbar",
    step_type=Fileset,
    builder=get_fileset,
    builder_params={"with_failure": True,
                    "n_files_max_per_sample": 2},  # -1 for the full AGC
    output="fileset_dict",
)
step_analysis = Step(
    name="Analysis_ttbar",
    step_type=Analysis,
    builder=run_analysis,
    builder_params={"use_inference": False},       # ML inference needs xgboost + models/
    input="fileset_dict",
    output="analysis_payload",
)
step_plotting = Step(
    name="Plot_ttbar",
    step_type=Plotting,
    builder=plotting_1,
    input="analysis_payload",
)

workflow = Workflow()
workflow.add(step_fileset)
workflow.add(step_analysis, depends_on=[step_fileset])
workflow.add(step_plotting, depends_on=[step_analysis])

Step(name='Plot_ttbar', step_type=<class 'coffea_workflow.artifacts.Plotting'>, builder=<function plotting_1 at 0x7fddb8a2a160>, builder_params=None, facility=None, executor_config=None, input='analysis_payload', output=None)

## Facility

The factory does the boilerplate: Dask `Client` construction, package installation on workers (`worker_packages`), and uploading the analysis module and `utils/` (`worker_files`). It is closed automatically at the end of `run()`.

In [3]:
facility = facilities.CoffeaCasaFactory(
    worker_packages=("coffea>=2026.7.0", "correctionlib", "vector"),
    worker_files=("ttbar_analysis.py", "utils"),
)

## Run(demo-size)

The run is kept small with the workflow's own knobs, no code changes: `datasets` restricts to the corrupted dataset plus the two smallest ones (≈4.4M events instead of 19.5M), and `percentage=50` makes **single-file chunks** — 6 in total. 

In [ ]:
config = RunConfig(
    strategy="by_dataset",
    percentage=50,                             # 1-file chunks: failure isolated to one file
    datasets=["ttbar__nominal",                # the dataset with the broken URL
              "single_top_s_chan__nominal",    # two smallest datasets
              "single_top_tW__nominal"],
    cache_dir=".cache_coffea_casa_failure",
    facility=facility,
    executor_config=ExecutorConfig(executor_type="DaskExecutor"),
)

result = run(workflow, config)

execution took 1.80 secondsERROR! Session/line number was not unique in database. History logging moved to new 
session 820

Workflow DAG:
  [0] Fileset_ttbar -> Fileset builder=<function get_fileset at 0x7fddb9e071a0>
  [1] Analysis_ttbar -> Analysis builder=<function run_analysis at 0x7fddb907da80>
  [2] Plot_ttbar -> Plotting builder=<function plotting_1 at 0x7fddb8a2a160>
Edges:
  Fileset_ttbar -> Analysis_ttbar
  Analysis_ttbar -> Plot_ttbar

Run config:
  Strategy:  by_dataset
  Executor:  DaskExecutor  workers=None
  Facility:  CoffeaCasaFactory

Executing step 'Fileset_ttbar' of type 'Fileset' with the user code <function get_fileset at 0x7fddb9e071a0> and user parameters {'with_failure': True, 'n_files_max_per_sample': 2}
Extracted from cache: .cache_coffea_casa_failure/Fileset/9851a207a8f82975ed75bbbbdb07327c30f954a4084eb2371749da19b72d3d90
  -> materialized at .cache_coffea_casa_failure/Fileset/9851a207a8f82975ed75bbbbdb07327c30f954a4084eb2371749da19b72d3d90

Executing step 'Analysis_ttbar' of type 'Analysis' with the user code <function run_analysis at 0x7fddb907da80> and user parameters {'use_in

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ /usr/local/lib/python3.12/site-packages/distributed/worker.py:2982 in _run_task_simple           │
│                                                                                                  │
│   2979 │   │   context_meter.meter("thread-cpu", func=thread_time),                              │
│   2980 │   ):                                                                                    │
│   2981 │   │   try:                                                                              │
│ ❱ 2982 │   │   │   result = task(data)                                                           │
│   2983 │   │   except (SystemExit, KeyboardInterrupt):                                           │
│   2984 │   │   │   # Special-case these, just like asyncio does all over the place. They will    │
│   2985 │   │   │   # pass through `fail_hard` and `_handle_stimulus_from_task`, and eventually   │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/dask/_task_spec.py:755 in __call__                       │
│                                                                                                  │
│    752 │   │   │   │   for k, kw in self.kwargs.items()                                          │
│    753 │   │   │   }                                                                             │
│    754 │   │   │   return self.func(*new_argspec, **kwargs)                                      │
│ ❱  755 │   │   return self.func(*new_argspec)                                                    │
│    756 │                                                                                         │
│    757 │   def __setstate__(self, state):                                                        │
│    758 │   │   slots = self.__class__.get_all_slots()                                            │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1290 in automatic_retries   │
│                                                                                                  │
│   1287 │   │   │   │   │   if use_result_type:                                                   │
│   1288 │   │   │   │   │   │   # surface the exception instead of silently skipping              │
│   1289 │   │   │   │   │   │   # so the Runner can wrap it as Err                                │
│ ❱ 1290 │   │   │   │   │   │   raise e                                                           │
│   1291 │   │   │   │   │   warnings.warn(                                                        │
│   1292 │   │   │   │   │   │   f"Skipping bad file after {retry_count + 1} attempts. The last e  │
│   1293 │   │   │   │   │   )                                                                     │
│                                                                                                  │
│ /usr/local/lib/python3.12/site-packages/coffea/processor/executor.py:1276 in automatic_retries   │
│                                                                                                  │
│   1273 │   │   retry_count = 0                                                                   │
│   1274 │   │   while retry_count <= retries:                                                     │
│   1275 │   │   │   try:                                                                          │
│ ❱ 1276 │   │   │   │   return func(*args, **kwargs)                                              │
│   1277 │   │   │   except Exception as e:                                                        │
│   1278 │   │   │   │   warnings.warn(                                                            │
│   1279 │   │   │   │   │   f"Performed attempt {retry_count